<a href="https://colab.research.google.com/github/eceirem/COVID19-Pneumonia-XRay-Classification/blob/main/notebooks/01_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
01_preprocessing.py
Author: Ece
Description: This script downloads the COVID-19 Radiography Database, pairs original X-ray
images with their corresponding lung masks, applies a bitwise mask to remove background noise
(bones, medical equipment), and enhances the contrast using CLAHE. The final images are
saved in a 'Preprocessed' directory for downstream Machine Learning and Deep Learning tasks.
"""

import os
import cv2
import glob
from tqdm import tqdm # For progress bar

# Configuration and Paths
DATASET_PATH = "COVID-19_Radiography_Dataset"
OUTPUT_BASE_DIR = "Dataset/Preprocessed"
CLASSES = ["COVID", "Normal", "Viral Pneumonia"]
TARGET_SIZE = (256, 256) # Standardizing image size for DL/ML models to prevent RAM crashes

def apply_mask_and_clahe(img_path, mask_path, output_path):
    """
    Reads the image and mask, applies the mask via Bitwise AND, applies CLAHE,
    and saves the preprocessed image.
    """
    # 1. Read image and mask in grayscale
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    if img is None or mask is None:
        print(f"[WARNING] Could not read image or mask: {img_path}")
        return

    # Resize both to ensure they match perfectly (and to save RAM later)
    img = cv2.resize(img, TARGET_SIZE)
    mask = cv2.resize(mask, TARGET_SIZE)

    # 2. Binarize the mask to ensure it strictly contains 0 or 255
    _, binary_mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

    # 3. Apply the mask (Background becomes 0/black, lungs remain 100% original)
    masked_img = cv2.bitwise_and(img, img, mask=binary_mask)

    # 4. Apply CLAHE (Contrast Limited Adaptive Histogram Equalization)
    # clipLimit=2.0 and tileGridSize=(8,8) are standard values in medical imaging literature
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced_img = clahe.apply(masked_img)

    # 5. Save the final image
    cv2.imwrite(output_path, enhanced_img)

def process_dataset():
    """
    Iterates through all classes, matches images with masks, and processes them.
    """
    print("[INFO] Starting Data Preprocessing Pipeline...")

    for cls in CLASSES:
        img_dir = os.path.join(DATASET_PATH, cls, "images")
        mask_dir = os.path.join(DATASET_PATH, cls, "masks")
        output_dir = os.path.join(OUTPUT_BASE_DIR, cls)

        # Create output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)

        # Get all image paths
        img_paths = glob.glob(os.path.join(img_dir, "*.png"))

        print(f"\n[INFO] Processing class: {cls} ({len(img_paths)} images found)")

        # Wrap with tqdm for a nice progress bar in Colab
        for img_path in tqdm(img_paths, desc=f"Processing {cls}"):
            # Extract filename (e.g., COVID-1.png)
            filename = os.path.basename(img_path)
            mask_path = os.path.join(mask_dir, filename)

            # Define where to save the preprocessed image
            output_path = os.path.join(output_dir, filename)

            # Process and save if the mask exists
            if os.path.exists(mask_path):
                apply_mask_and_clahe(img_path, mask_path, output_path)
            else:
                print(f"[WARNING] Mask not found for {filename}. Skipping.")

if __name__ == "__main__":
    # --- KAGGLE DOWNLOAD SECTION (Uncomment in Colab) ---
    print("[INFO] Downloading Dataset from Kaggle...")
    !pip install -q kaggle
    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d tawsifurrahman/covid19-radiography-database
    !unzip -q covid19-radiography-database.zip

    # Run the preprocessing pipeline
    process_dataset()

    print("\n[SUCCESS] All images have been preprocessed and saved to Dataset/Preprocessed/")
    # In Colab, you can now zip this folder and send it to Doğukan:
    !zip -r Preprocessed_Dataset.zip Dataset/Preprocessed/

[INFO] Downloading Dataset from Kaggle...
cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/tawsifurrahman/covid19-radiography-database
License(s): copyright-authors
100% 778M/778M [00:03<00:00, 251MB/s]

[INFO] Starting Data Preprocessing Pipeline...

[INFO] Processing class: COVID (3616 images found)


Processing COVID: 100%|██████████| 3616/3616 [00:11<00:00, 301.79it/s]



[INFO] Processing class: Normal (10192 images found)


Processing Normal: 100%|██████████| 10192/10192 [00:33<00:00, 302.71it/s]



[INFO] Processing class: Viral Pneumonia (1345 images found)


Processing Viral Pneumonia: 100%|██████████| 1345/1345 [00:04<00:00, 293.23it/s]


Görüntülenen çıkış son 5000 satıra kısaltıldı.
  adding: Dataset/Preprocessed/Normal/Normal-1480.png (deflated 4%)
  adding: Dataset/Preprocessed/Normal/Normal-1116.png (deflated 2%)
  adding: Dataset/Preprocessed/Normal/Normal-5549.png (deflated 3%)
  adding: Dataset/Preprocessed/Normal/Normal-4629.png (deflated 4%)
  adding: Dataset/Preprocessed/Normal/Normal-2937.png (deflated 3%)
  adding: Dataset/Preprocessed/Normal/Normal-6896.png (deflated 2%)
  adding: Dataset/Preprocessed/Normal/Normal-3651.png (deflated 2%)
  adding: Dataset/Preprocessed/Normal/Normal-3938.png (deflated 3%)
  adding: Dataset/Preprocessed/Normal/Normal-6344.png (deflated 2%)
  adding: Dataset/Preprocessed/Normal/Normal-8523.png (deflated 3%)
  adding: Dataset/Preprocessed/Normal/Normal-1912.png (deflated 3%)
  adding: Dataset/Preprocessed/Normal/Normal-1930.png (deflated 3%)
  adding: Dataset/Preprocessed/Normal/Normal-8206.png (deflated 2%)
  adding: Dataset/Preprocessed/Normal/Normal-7062.png (deflated 3%)
 

In [ ]:
import os
import random
import shutil

source_dir = "Dataset/Preprocessed"
target_dir = "Dataset/Balanced_Dogukan"
classes = ["COVID", "Normal", "Viral Pneumonia"]

# Find the number of samples in the minority class (Viral Pneumonia = 1345)
min_samples = min([len(os.listdir(os.path.join(source_dir, c))) for c in classes])

print(f"[INFO] Selecting {min_samples} images from each class...")
os.makedirs(target_dir, exist_ok=True)

for cls in classes:
    src_class_dir = os.path.join(source_dir, cls)
    target_class_dir = os.path.join(target_dir, cls)
    os.makedirs(target_class_dir, exist_ok=True)

    # List all images and randomly select 'min_samples' amount
    all_images = os.listdir(src_class_dir)
    selected_images = random.sample(all_images, min_samples)

    # Copy the selected images to Dogukan's directory
    for img in selected_images:
        shutil.copy(os.path.join(src_class_dir, img), os.path.join(target_class_dir, img))

    print(f"--> Copied {min_samples} images from class: {cls}")

# Create a zip file for Dogukan
print("[INFO] Creating the zip file for Dogukan...")
!zip -r -q Balanced_Dataset_Dogukan.zip Dataset/Balanced_Dogukan/
print("[SUCCESS] Balanced_Dataset_Dogukan.zip is ready!")

[INFO] Selecting 1345 images from each class...
--> Copied 1345 images from class: COVID
--> Copied 1345 images from class: Normal
--> Copied 1345 images from class: Viral Pneumonia
[INFO] Creating the zip file for Dogukan...
[SUCCESS] Balanced_Dataset_Dogukan.zip is ready!
